In [1]:
import pandas as pd
import re

In [2]:
Dataset = pd.read_csv(
    "UK-Sanctions-List.csv",
    skiprows=1
)
print(Dataset.head())

print(Dataset.shape)
print(Dataset.columns.tolist())

  Last Updated Unique ID  OFSI Group ID UN Reference Number  \
0   16/04/2026   AFG0001        12703.0             TAe.010   
1   16/04/2026   AFG0001        12703.0             TAe.010   
2   16/04/2026   AFG0001        12703.0             TAe.010   
3   16/04/2026   AFG0001        12703.0             TAe.010   
4   16/04/2026   AFG0001        12703.0             TAe.010   

                                       Name 6 Name 1 Name 2 Name 3 Name 4  \
0  HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE    NaN    NaN    NaN    NaN   
1  HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE    NaN    NaN    NaN    NaN   
2  HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE    NaN    NaN    NaN    NaN   
3  HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE    NaN    NaN    NaN    NaN   
4  HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE    NaN    NaN    NaN    NaN   

  Name 5  ... IMO number Current owner/operator (s)  \
0    NaN  ...        NaN                        NaN   
1    NaN  ...        NaN                        

/var/folders/52/qj9xhmk16wncspp1yf0l67bm0000gn/T/ipykernel_56984/1760453406.py:1: DtypeWarning: Columns (48,49,50,51,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  Dataset = pd.read_csv(


In [3]:
name_cols = ["Name 1", "Name 2", "Name 3", "Name 4", "Name 5", "Name 6"]

Dataset["Name_combined"] = (
    Dataset[name_cols]
    .fillna("")                         # replace NaNs with empty strings
    .agg(" ".join, axis=1)              # join across columns
    .str.replace(r"\s+", " ", regex=True)  # clean extra spaces
    .str.strip()
)

In [4]:
address_cols = [ "Address Line 1", "Address Line 2", "Address Line 3", "Address Line 4", "Address Line 5", "Address Line 6", "Address Postal Code"]

Dataset["Address_Combined"] = (
    Dataset[address_cols]
    .fillna("")  
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Treat empty addresses as NaN for cleaner counting
Dataset["Address_Combined"] = Dataset["Address_Combined"].replace("", pd.NA)

In [5]:
def clean_dob(dob):
    if pd.isna(dob) or str(dob).strip() == "":
        return "Unknown"
    
    dob = str(dob).strip()
    
    # Split into parts
    parts = dob.split("/")
    
    # Handle different formats safely
    if len(parts) == 3:
        day, month, year = parts
        
        # Case: only year known
        if day.lower() == "dd" and month.lower() == "mm":
            return year
        
        # Case: month + year known
        if day.lower() == "dd":
            return f"{month}/{year}"
        
        # Case: full date known
        return dob
    
    return dob

In [6]:
reduced = Dataset[[
    "Unique ID",
    "Last Updated",
    "Name_combined",
    "Gender",
    "Address_Combined",
    "Name type",
    "Designation source",
    "Designation Type",
    "D.O.B",
    "Address Country",
    "Country of birth",
    "Nationality(/ies)",
    "National Identifier number",
    "Passport number",
    "Sanctions Imposed",
    "Other Information"
]].copy()

reduced.columns = reduced.columns.str.strip()

# Clean missing values
for col in reduced.columns:
    if reduced[col].dtype == "object":
        reduced[col] = (
            reduced[col]
            .fillna("Unknown")
            .replace(r"^\s*$", "Unknown", regex=True))
        
#Explicitly clean D.O.B to handle empty strings and NaNs
reduced["D.O.B"] = reduced["D.O.B"].apply(clean_dob)

# Group by Unique ID, Name, and D.O.B, to form unique entrie, then aggregate other fields
reduced = (
    reduced.groupby(["Unique ID", "Name_combined", "D.O.B"], as_index=False)
    .agg(
        **{
            "Gender": ("Gender", "first"),
            "Name Type": ("Name type", "first"),
            "Designation Source": ("Designation source", "first"),
            "Designation Type": ("Designation Type", "first"),
            "Last Updated": ("Last Updated", "first"),
            
            "Address Country": ("Address Country",
                lambda x: ", ".join(sorted(set([str(v).strip() for v in x if str(v).strip() != "Unknown"]))) or "Unknown"),
            "Country of birth": ("Country of birth",
                lambda x: ", ".join(sorted(set([str(v).strip() for v in x if str(v).strip() != "Unknown"]))) or "Unknown"),
            "Nationality(/ies)": ("Nationality(/ies)",
                lambda x: ", ".join(sorted(set([str(v).strip() for v in x if str(v).strip() != "Unknown"]))) or "Unknown"),

            "National Identifier number": ("National Identifier number", "first"),
            "Passport Number": ("Passport number", "first"),
            "Sanctions Imposed": ("Sanctions Imposed", "first"),
            "Other Information": ("Other Information", "first"),
            "Address Variations": ("Address_Combined", "nunique")
        }
    )
)

# Final rename (clean output)
reduced = reduced.rename(columns={
    "Name_combined": "Name",
    "D.O.B": "Date of Birth",
    "Country of birth": "Birth Country",
    "Nationality(/ies)": "Nationalities"
})

cols_to_fix = ["Address Country", "Birth Country", "Nationalities"]

In [7]:
#Save reducded dataser:
reduced.to_csv("UK_Sanctions_Reduced.csv", index=False, encoding="utf-8")